In [1]:
import mediapipe as mp
import cv2
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import precision_score, accuracy_score, f1_score, recall_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

### 1. Train Model

#### 1.1. Describe data and split dataset

In [2]:
def rescale_frame(frame, percent=50):
    '''
    Rescale a frame to a certain percentage compare to its original frame
    '''
    width = int(frame.shape[1] * percent/ 100)
    height = int(frame.shape[0] * percent/ 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation = cv2.INTER_AREA)


def describe_dataset(dataset_path: str):
    '''
    Describe dataset
    '''

    data = pd.read_csv(dataset_path)
    print(f"Headers: {list(data.columns.values)}")
    print(f'Number of rows: {data.shape[0]} \nNumber of columns: {data.shape[1]}\n')
    print(f"Labels: \n{data['label'].value_counts()}\n")
    print(f"Missing values: {data.isnull().values.any()}\n")
    
    duplicate = data[data.duplicated()]
    print(f"Duplicate Rows : {len(duplicate.sum(axis=1))}")

    return data


def round_up_metric_results(results) -> list:
    '''Round up metrics results such as precision score, recall score, ...'''
    return list(map(lambda el: round(el, 3), results))

In [3]:
# df = describe_dataset("./train.csv")
# df = describe_dataset("train_augmented.csv")
df = describe_dataset("train_augmented.csv")
# df.loc[df["label"] == "c", "label"] = 0
# df.loc[df["label"] == "i", "label"] = 1
# df.loc[df["label"] == "H", "label"] = 2
# df.loc[df["label"] == "L", "label"] = 3

label_map = {
    'c': 0,                  # correct pose
    'feet_too_close': 1,     # stance too narrow
    'arms_dropped': 2,       # arms hanging too low
    'shoulders_raised': 3,   # shoulders tense/raised
    'leaning_forward': 4,    # torso leaning forward
    'leaning_back': 5,       # torso leaning backward
    'head_tilted': 6         # head tilted instead of neutral
}


df["label"] = df["label"].map(label_map)
reverse_map = {v: k for k, v in label_map.items()}



df.head(3)

Headers: ['label', 'nose_x', 'nose_y', 'nose_z', 'nose_v', 'left_ear_x', 'left_ear_y', 'left_ear_z', 'left_ear_v', 'right_ear_x', 'right_ear_y', 'right_ear_z', 'right_ear_v', 'left_shoulder_x', 'left_shoulder_y', 'left_shoulder_z', 'left_shoulder_v', 'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z', 'right_shoulder_v', 'left_elbow_x', 'left_elbow_y', 'left_elbow_z', 'left_elbow_v', 'right_elbow_x', 'right_elbow_y', 'right_elbow_z', 'right_elbow_v', 'left_wrist_x', 'left_wrist_y', 'left_wrist_z', 'left_wrist_v', 'right_wrist_x', 'right_wrist_y', 'right_wrist_z', 'right_wrist_v', 'left_index_x', 'left_index_y', 'left_index_z', 'left_index_v', 'right_index_x', 'right_index_y', 'right_index_z', 'right_index_v', 'left_hip_x', 'left_hip_y', 'left_hip_z', 'left_hip_v', 'right_hip_x', 'right_hip_y', 'right_hip_z', 'right_hip_v', 'left_knee_x', 'left_knee_y', 'left_knee_z', 'left_knee_v', 'right_knee_x', 'right_knee_y', 'right_knee_z', 'right_knee_v', 'left_ankle_x', 'left_ankle_y', '

,label,nose_x,nose_y,nose_z,nose_v,left_ear_x,left_ear_y,left_ear_z,left_ear_v,right_ear_x,...,right_ankle_z,right_ankle_v,left_heel_x,left_heel_y,left_heel_z,left_heel_v,right_heel_x,right_heel_y,right_heel_z,right_heel_v
0,0,0.577521,0.172172,-0.766958,0.999985,0.621025,0.166568,-0.446601,0.999918,0.521230,...,0.561316,0.970042,0.559702,0.801190,0.577578,0.808027,0.513515,0.795172,0.594650,0.791788
1,0,0.516222,0.182567,-0.848024,0.999997,0.575722,0.175204,-0.556826,0.999970,0.469550,...,0.502810,0.982540,0.547374,0.852726,0.529477,0.920109,0.490104,0.838644,0.529114,0.867764
2,0,0.511727,0.252431,-0.285891,0.999978,0.551945,0.249360,-0.135423,0.999814,0.467546,...,0.270659,0.983588,0.534532,0.769787,0.269265,0.898792,0.414116,0.765280,0.285243,0.864618


In [4]:
# Extract features and class
X = df.drop("label", axis=1)
y = df["label"].astype("int")

In [5]:
sc = StandardScaler()
# X = pd.DataFrame(sc.fit_transform(X))
X = pd.DataFrame(sc.fit_transform(X), columns=X.columns, index=X.index)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)
y_test.head(3)

6642    1
207     6
7545    0
Name: label, dtype: int32

#### 1.2. Train model using Scikit-Learn and train set evaluation

In [7]:
algorithms =[("LR", LogisticRegression()),
         ("SVC", SVC(probability=True)),
         ('KNN',KNeighborsClassifier()),
         ("DTC", DecisionTreeClassifier()),
         ("SGDC", CalibratedClassifierCV(SGDClassifier())),
         ("NB", GaussianNB()),
         ('RF', RandomForestClassifier()),]

models = {}
final_results = []

for name, model in algorithms:
    trained_model = model.fit(X_train, y_train)
    models[name] = trained_model

    # Evaluate model
    model_results = model.predict(X_test)

    p_score = precision_score(y_test, model_results, average=None, labels=[0, 1, 2,3,4,5,6])
    a_score = accuracy_score(y_test, model_results)
    r_score = recall_score(y_test, model_results, average=None, labels=[0, 1, 2,3,4,5,6])
    f1_score_result = f1_score(y_test, model_results, average=None, labels=[0, 1, 2,3,4,5,6])
    cm = confusion_matrix(y_test, model_results, labels=[0, 1, 2,3,4,5,6])
    final_results.append(( name,  round_up_metric_results(p_score), a_score, round_up_metric_results(r_score), round_up_metric_results(f1_score_result), cm))


In [9]:
# Sort results by F1 score
final_results.sort(key=lambda k: sum(k[4]), reverse=True)

pd.DataFrame(final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])

,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,RF,"[0.985, 0.995, 1.0, 0.996, 0.995, 1.0, 0.996]",0.995386,"[0.985, 0.991, 1.0, 1.0, 1.0, 0.995, 0.996]","[0.985, 0.993, 1.0, 0.998, 0.998, 0.998, 0.996]","[[194, 1, 0, 0, 1, 0, 1], [2, 221, 0, 0, 0, 0,..."
1,LR,"[0.97, 0.995, 1.0, 1.0, 1.0, 1.0, 1.0]",0.995386,"[0.995, 0.987, 1.0, 1.0, 0.995, 0.995, 0.996]","[0.982, 0.991, 1.0, 1.0, 0.998, 0.998, 0.998]","[[196, 1, 0, 0, 0, 0, 0], [3, 220, 0, 0, 0, 0,..."
2,SGDC,"[0.98, 0.982, 1.0, 1.0, 1.0, 1.0, 1.0]",0.994726,"[0.985, 0.991, 1.0, 0.995, 1.0, 0.991, 1.0]","[0.982, 0.987, 1.0, 0.998, 1.0, 0.995, 1.0]","[[194, 3, 0, 0, 0, 0, 0], [2, 221, 0, 0, 0, 0,..."
3,SVC,"[0.87, 0.995, 1.0, 0.978, 0.995, 1.0, 0.986]",0.974291,"[0.985, 0.969, 1.0, 0.986, 0.995, 0.986, 0.906]","[0.924, 0.982, 1.0, 0.982, 0.995, 0.993, 0.945]","[[194, 0, 0, 0, 1, 0, 2], [6, 216, 0, 1, 0, 0,..."
4,DTC,"[0.912, 1.0, 0.995, 0.956, 0.986, 0.986, 0.966]",0.971655,"[0.944, 0.982, 0.986, 0.982, 0.982, 0.967, 0.957]","[0.928, 0.991, 0.99, 0.969, 0.984, 0.976, 0.962]","[[186, 0, 1, 4, 2, 0, 4], [2, 219, 0, 1, 0, 0,..."
5,KNN,"[0.689, 0.962, 1.0, 0.915, 1.0, 0.99, 0.932]",0.916941,"[0.944, 0.906, 0.981, 0.923, 0.982, 0.939, 0.762]","[0.797, 0.933, 0.99, 0.919, 0.991, 0.964, 0.838]","[[186, 3, 0, 3, 0, 0, 5], [15, 202, 0, 2, 0, 1..."
6,NB,"[0.511, 0.738, 0.905, 0.705, 0.858, 0.817, 0.451]",0.696111,"[0.122, 0.695, 0.91, 0.658, 0.835, 0.778, 0.821]","[0.197, 0.716, 0.907, 0.681, 0.847, 0.797, 0.582]","[[24, 21, 9, 19, 9, 9, 106], [7, 155, 5, 7, 10..."


#### 1.3. Test set evaluation

In [ ]:
# test_df = describe_dataset("./test.csv")
test_df = describe_dataset("test.csv")
test_df = test_df.sample(frac=1).reset_index(drop=True)

test_df.loc[test_df["label"] == "C", "label"] = 0
test_df.loc[test_df["label"] == "H", "label"] = 1
test_df.loc[test_df["label"] == "L", "label"] = 2

test_x = test_df.drop("label", axis=1)
test_y = test_df["label"].astype("int")

test_x = pd.DataFrame(sc.transform(test_x))

In [ ]:
# Evaluate models on the test sets
testset_final_results = []

for name, model in models.items():
    # Evaluate model
    model_results = model.predict(test_x)

    p_score = precision_score(test_y, model_results, average=None, labels=[0, 1, 2])
    a_score = accuracy_score(test_y, model_results)
    r_score = recall_score(test_y, model_results, average=None, labels=[0, 1, 2])
    f1_score_result = f1_score(test_y, model_results, average=None, labels=[0, 1, 2])
    cm = confusion_matrix(test_y, model_results, labels=[0, 1, 2])
    testset_final_results.append(( name,  round_up_metric_results(p_score), a_score, round_up_metric_results(r_score), round_up_metric_results(f1_score_result), cm ))


testset_final_results.sort(key=lambda k: sum(k[4]), reverse=True)
pd.DataFrame(testset_final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])

In [9]:
# Evaluate LR model on test set
lr_model = models["LR"]
lr_predictions = lr_model.predict(X_test)

lr_precision = precision_score(y_test, lr_predictions, average=None, labels=[0, 1, 2,3,4,5,6,7])
lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_recall = recall_score(y_test, lr_predictions, average=None, labels=[0, 1, 2,3,4,5,6,7])
lr_f1 = f1_score(y_test, lr_predictions, average=None, labels=[0, 1, 2,3,4,5,6,7])

print("LR Precision:", round_up_metric_results(lr_precision))
print("LR Accuracy:", lr_accuracy)
print("LR Recall:", round_up_metric_results(lr_recall))
print("LR F1 Score:", round_up_metric_results(lr_f1))

LR Precision: [0.97, 0.995, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0]
LR Accuracy: 0.995385629531971
LR Recall: [0.995, 0.987, 1.0, 1.0, 0.995, 0.995, 0.996, 0.0]
LR F1 Score: [0.982, 0.991, 1.0, 1.0, 0.998, 0.998, 0.998, 0.0]


#### 1.4. Dumped model and input scaler using pickle

According to the evaluations, there are multiple good models at the moment, therefore, the best models are LR and Ridge.

In [10]:
with open("./model/all_sklearn.pkl", "wb") as f:
    pickle.dump(models, f)

In [11]:
with open("./model/LR_model.pkl", "wb") as f:
    pickle.dump(models["LR"], f)

In [12]:
with open("./model/SVC_model.pkl", "wb") as f:
    pickle.dump(models["SVC"], f)

In [13]:
# Dump input scaler
with open("./model/input_scaler.pkl", "wb") as f:
    pickle.dump(sc, f)